In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from time import perf_counter
import inspect
import re
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM

In [3]:
PROJECT = Path("/content/drive/MyDrive/Vision/vision_unit_04")
MANIFEST = PROJECT / "outputs/evaluation_manifest.csv"
DATA = Path("/content/datasets/coco128-seg")

manifest = pd.read_csv(MANIFEST)
cal_names = manifest.loc[manifest.split == "calibration", "image_name"].tolist()
test_names = manifest.loc[manifest.split == "test", "image_name"].tolist()

print(f"Calibration: {len(cal_names)} images | Test: {len(test_names)} images")

Calibration: 20 images | Test: 20 images


In [5]:
CLASSES = {0: "person", 2: "car", 16: "dog", 41: "cup", 56: "chair"}
NAME_TO_ID = {name: cid for cid, name in CLASSES.items()}
PROMPTS = {
    "combined": ("chair. cup. dog. car. person.",),
    "separate": ("chair.", "cup.", "dog.", "car.", "person."),
}
BOX_THRESHOLD, TEXT_THRESHOLD = 0.35, 0.30
MODEL_ID = "IDEA-Research/grounding-dino-tiny"

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID)
detector = detector.to("cuda").float().eval()
sam = SAM("sam2.1_s.pt")

postprocess = processor.post_process_grounded_object_detection
box_argument = (
    "threshold" if "threshold" in inspect.signature(postprocess).parameters
    else "box_threshold"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

**Read Image and GT**

In [7]:
def load_sample(name):
    image = Image.open(DATA / "images/train2017" / name).convert("RGB")
    width, height = image.size
    
    label_path = DATA / "labels/train2017" / f"{Path(name).stem}.txt"
    gt = []

    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        
        values = list(map(float, line.split()))
        cid = int(values[0])
        if cid not in CLASSES:
            continue
        
        points = np.array(values[1:]).reshape(-1, 2) * [width, height]
        points = np.clip(points, [0, 0], [width - 1, height - 1])
        canvas = Image.new("L", image.size, 0)
        
        ImageDraw.Draw(canvas).polygon([tuple(p) for p in points], fill=1)
        mask = np.asarray(canvas, dtype=bool)
        ys, xs = np.nonzero(mask)
        
        if len(xs):
            gt.append({
                "class_id": cid, "mask": mask,
                "box": np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]),
            })
            
    return image, gt


**Text → boxes**

In [ ]:
@torch.inference_mode()
def detect(image, prompt, box_threshold, text_threshold):
    inputs = processor(
        images=image, text=prompt, return_tensors="pt"
    ).to("cuda")
    
    outputs = detector(**inputs)
    result = postprocess(
        outputs, inputs.input_ids,
        **{box_argument: box_threshold},
        text_threshold=text_threshold,
        target_sizes=[image.size[::-1]]
    )[0]
    
    labels = (
        result["text_labels"] if "text_labels" in result
        else result["labels"]
    )
    
    return {
        "boxes": result["boxes"].cpu().numpy().reshape(-1, 4),
        "scores": result["scores"].cpu().numpy(),
        "labels": list(labels)
    }

**Reusable grounded inference**

In [14]:
def grounded_predict(image, prompts, box_threshold=0.35, text_threshold=0.30):
    image = image.convert("RGB")
    torch.cuda.synchronize()
    start = perf_counter()
    parts = [detect(image, p, box_threshold, text_threshold) for p in prompts]
    
    prediction = {
        "boxes": np.concatenate([p["boxes"] for p in parts]),
        "scores": np.concatenate([p["scores"] for p in parts]),
        "labels": [label for p in parts for label in p["labels"]],
    }
    
    boxes = prediction["boxes"]
    if len(boxes):
        result = sam(
            image, bboxes=boxes.tolist(), device=0, imgsz=1024,
            retina_masks=True, verbose=False,
        )[0]
        masks = result.masks.data.cpu().numpy() > 0.5
    else:
        masks = np.zeros((0, image.height, image.width), dtype=bool)


    torch.cuda.synchronize()
    prediction["total_ms"] = (perf_counter() - start) * 1000
    prediction["masks"] = masks
    return prediction

In [10]:
def normalize_label(label):
    text = re.sub(r"[^a-z0-9\s]", " ", label.lower())
    return " ".join(w for w in text.split() if w not in {"a", "an", "the"})

def box_iou(a, b):
    overlap = np.maximum(0, np.minimum(a[2:], b[2:]) - np.maximum(a[:2], b[:2]))
    intersection = np.prod(overlap)
    area_a = np.prod(np.maximum(0, a[2:] - a[:2]))
    area_b = np.prod(np.maximum(0, b[2:] - b[:2]))
    return float(intersection / max(area_a + area_b - intersection, 1e-9))

**evaluation rule**

In [11]:
def evaluate(prediction, gt):
    status = np.zeros(len(gt), dtype=np.uint8)
    labels = [normalize_label(label) for label in prediction["labels"]]
    
    for i in np.argsort(-prediction["scores"]):
        cid = NAME_TO_ID.get(labels[i])
        
        candidates = [
            (box_iou(prediction["boxes"][i], item["box"]), j)
            for j, item in enumerate(gt)
            if item["class_id"] == cid and status[j] == 0
        ]
        overlap, j = max(candidates, default=(0.0, -1))
        
        if overlap < 0.50:
            continue
        
        mask, target = prediction["masks"][i], gt[j]["mask"]
        mask_iou = np.logical_and(mask, target).sum() / max(
            np.logical_or(mask, target).sum(), 1
        )
        status[j] = 2 if mask_iou >= 0.50 else 1
        
    tp = int((status > 0).sum())
    
    return {
        "gt": len(gt), "tp": tp, "fp": len(labels) - tp,
        "good_masks": int((status == 2).sum()),
        "unmapped_fp": sum(label not in NAME_TO_ID for label in labels),
        "total_ms": prediction["total_ms"]
    }, status

In [12]:
def summarize(rows):
    frame = pd.DataFrame(rows)
    gt, tp, fp, good, unmapped = frame[
        ["gt", "tp", "fp", "good_masks", "unmapped_fp"]
    ].sum()
    return {
        "gt": int(gt), "tp": int(tp), "fp": int(fp), "fn": int(gt - tp),
        "unmapped_fp": int(unmapped), "good_masks": int(good),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(gt, 1),
        "f1": 2 * tp / max(gt + tp + fp, 1),
        "conditional_mask_success": good / tp if tp else np.nan,
        "mask_recall": good / max(gt, 1),
        "mean_ms": frame.total_ms.mean(),
        "p95_ms": np.percentile(frame.total_ms, 95)
    }

**Compare and configuration**

In [15]:
warm_image, _ = load_sample(cal_names[0])
for prompts in PROMPTS.values():
    _ = grounded_predict(warm_image, prompts, BOX_THRESHOLD, TEXT_THRESHOLD)

cal_rows = {mode: [] for mode in PROMPTS}

for index, name in enumerate(tqdm(cal_names, desc="Calibration")):
    image, gt = load_sample(name)
    order = list(PROMPTS) if index % 2 == 0 else list(PROMPTS)[::-1]
    for mode in order:
        prediction = grounded_predict(
            image, PROMPTS[mode], BOX_THRESHOLD, TEXT_THRESHOLD
        )
        row, _ = evaluate(prediction, gt)
        cal_rows[mode].append(row)

comparison = pd.DataFrame({
    mode: summarize(rows) for mode, rows in cal_rows.items()
}).T

display(comparison.round(3))


Calibration:   0%|          | 0/20 [00:00<?, ?it/s]

,gt,tp,fp,fn,unmapped_fp,good_masks,precision,recall,f1,conditional_mask_success,mask_recall,mean_ms,p95_ms
combined,110.0,40.0,27.0,70.0,15.0,39.0,0.597,0.364,0.452,0.975,0.355,785.613,925.854
separate,110.0,26.0,113.0,84.0,68.0,25.0,0.187,0.236,0.209,0.962,0.227,3110.769,3853.141


In [16]:
baseline, separate = comparison.loc["combined"], comparison.loc["separate"]
final_mode = (
    "separate"
    if separate.mask_recall > baseline.mask_recall and separate.f1 >= baseline.f1
    else "combined"
)
final_prompts = tuple(PROMPTS[final_mode])
final_box_threshold, final_text_threshold = BOX_THRESHOLD, TEXT_THRESHOLD

def predict_final(image, prompts=final_prompts, box=final_box_threshold,
                  text=final_text_threshold):
    return grounded_predict(image, prompts, box, text)

print("Selected:", final_mode)
print("Prompts:", final_prompts)
print("Thresholds:", final_box_threshold, final_text_threshold)

Selected: combined
Prompts: ('chair. cup. dog. car. person.',)
Thresholds: 0.35 0.3
